# Race Simulation Analysis

This notebook visualizes the results of the race simulation. 
The agent controls **Sebastian Vettel (VET)**.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
import numpy as np

# Configuration
AGENT_DRIVER = 'VET'
RESULTS_BASE_DIR = 'logs/RaceStrategy-v2/results'

def get_latest_results_dir(base_dir):
    dirs = glob.glob(os.path.join(base_dir, '*'))
    dirs = [d for d in dirs if os.path.isdir(d)]
    if not dirs:
        return None
    return max(dirs, key=os.path.getmtime)

results_dir = get_latest_results_dir(RESULTS_BASE_DIR)
print(f"Analyzing results from: {results_dir}")

: 

## Load Data

In [ ]:
def load_data(results_dir, suffix):
    files = glob.glob(os.path.join(results_dir, f"*_{suffix}.csv"))
    if not files:
        print(f"No {suffix} file found.")
        return None
    # Use the latest file if multiple exist (though usually one per run per type)
    latest_file = max(files, key=os.path.getmtime)
    print(f"Loading {latest_file}")
    return pd.read_csv(latest_file)

df_laptimes = load_data(results_dir, 'laptimes')
df_positions = load_data(results_dir, 'positions')

## Lap Times Analysis

In [ ]:
if df_laptimes is not None:
    # Melt for plotting
    df_melted = df_laptimes.melt(id_vars=['lap'], var_name='driver', value_name='lap_time')
    
    plt.figure(figsize=(14, 7))
    
    # Plot other drivers with low opacity
    sns.lineplot(data=df_melted[df_melted['driver'] != AGENT_DRIVER], 
                 x='lap', y='lap_time', hue='driver', alpha=0.3, legend='brief')
    
    # Plot agent driver with high opacity and thicker line
    agent_data = df_melted[df_melted['driver'] == AGENT_DRIVER]
    if not agent_data.empty:
        plt.plot(agent_data['lap'], agent_data['lap_time'], 
                 label=f"{AGENT_DRIVER} (Agent)", color='red', linewidth=2.5)
    
    plt.title("Lap Times per Driver")
    plt.xlabel("Lap")
    plt.ylabel("Lap Time (s)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Classification (Positions) Analysis

In [ ]:
if df_positions is not None:
    # Melt for plotting
    df_pos_melted = df_positions.melt(id_vars=['lap'], var_name='driver', value_name='position')
    
    plt.figure(figsize=(14, 7))
    
    # Plot other drivers
    sns.lineplot(data=df_pos_melted[df_pos_melted['driver'] != AGENT_DRIVER], 
                 x='lap', y='position', hue='driver', alpha=0.3, legend='brief')
    
    # Plot agent driver
    agent_pos = df_pos_melted[df_pos_melted['driver'] == AGENT_DRIVER]
    if not agent_pos.empty:
        plt.plot(agent_pos['lap'], agent_pos['position'], 
                 label=f"{AGENT_DRIVER} (Agent)", color='red', linewidth=3, marker='o', markersize=4)
    
    plt.title("Race Classification (Position) per Lap")
    plt.xlabel("Lap")
    plt.ylabel("Position")
    plt.gca().invert_yaxis()  # 1st place at top
    plt.yticks(range(1, 21))  # Assuming 20 drivers
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()